# 🖼️ Image Models - Validation Tests

## Overview

Validate **image generation and editing** through the Unified AI API (`/unified-ai`). All image providers are routed through a single OpenAI-style images surface and return OpenAI-shaped responses (`data[].b64_json`).

This notebook tests:
- **Backend Onboarding (FLUX & MAI)**: augment the azd-loaded `llm_backends_config` with dedicated `azure-flux` / `azure-mai` backends and redeploy the onboarding Bicep (`gpt-image` rides the existing `ai-foundry` backend)
- **Access Contract Provisioning**: Deploy an access contract via Bicep to obtain API credentials scoped to the image models
- **Deployment Discovery**: Verify the image models appear in `GET /unified-ai/deployments`
- **gpt-image generation (body model)**: `POST /unified-ai/v1/images/generations` with `model` in the body (Azure OpenAI / Foundry)
- **gpt-image generation (deployments path)**: `POST /unified-ai/openai/deployments/{model}/images/generations`
- **FLUX generation** (`azure-flux`, optional): native Black Forest Labs surface via `modelPath` slug
- **MAI generation** (`azure-mai`, optional): native Microsoft MAI surface
- **Missing-model error**: `POST /unified-ai/v1/images/edits` without `x-ai-model` returns `400 missing_model_parameter`
- **Image edit**: edit a generated image via `POST /unified-ai/openai/deployments/{model}/images/edits` (multipart)
- **RBAC enforcement**: a model outside the access contract's `allowedModels` returns `403`

> **Prerequisites:** An existing Citadel Governance Hub deployment with the Unified AI API enabled and at least one image model onboarded (see [LLM Backend Onboarding — Image Models](../bicep/infra/llm-backend-onboarding/README.md#image-models)).

### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group` and `location` from your active `azd` environment.
- `False` — fill the `REPLACE` values manually below.

Set the image model names to match your onboarded backends, and enable the `test_*` flags for the providers you have configured.

In [ ]:
import os
import sys, json, base64, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION (REQUIRED — used as defaults if azd lookup fails)
# ============================================================================
governance_hub_resource_group = "REPLACE"   # Resource group of the deployed Citadel Governance Hub
location = "REPLACE"                         # Azure region (e.g. "swedencentral", "eastus")
llm_backends_config = []                     # Existing backends (auto-loaded from azd LLM_BACKEND_CONFIG)

# ============================================================================
# 🖼️ IMAGE MODEL CONFIGURATION (REPLACE based on your onboarded backends)
# ============================================================================
openai_image_model = "gpt-image-1.5"          # Azure OpenAI / Foundry gpt-image model (ai-foundry / azure-openai)
flux_image_model   = "FLUX.2-pro"             # Black Forest Labs FLUX model (azure-flux); requires a modelPath slug onboarded
mai_image_model    = "MAI-Image-2.5-Flash"   # Microsoft MAI model (azure-mai)

# Feature toggles — enable only the providers you have configured
test_openai_image = True
test_flux         = True
test_mai          = True

# ============================================================================
# 🛠️ BACKEND ONBOARDING (FLUX & MAI need dedicated image backends)
# ----------------------------------------------------------------------------
# gpt-image-1.5 works out of the box on the existing ai-foundry backend (no
# special backend type). FLUX and MAI require dedicated azure-flux / azure-mai
# backends pointing at the Foundry resource's native image surfaces. The
# onboarding cell below augments your existing llm_backends_config (from azd)
# with those backends and deploys the LLM Backend Onboarding Bicep.
# ============================================================================
run_backend_onboarding = True               # Set False to skip the FLUX/MAI onboarding deployment
flux_image_model_path  = "flux-2-pro"        # BFL slug for flux_image_model (FLUX.2-pro -> flux-2-pro)
image_foundry_endpoint = ""                  # Override the FLUX/MAI base endpoint (https://<resource>.services.ai.azure.com/).
                                             # When empty, derived from an existing ai-foundry backend.

# A model name that is NOT granted by the access contract (used for the RBAC 403 test)
disallowed_image_model = "gpt-image-1"

# Prompt used across all generation tests
image_prompt = "A photograph of a red fox in an autumn forest, cinematic lighting"
save_images  = True   # Save decoded images next to this notebook for visual inspection

# ============================================================================
# 📜 ACCESS CONTRACT CONFIGURATION (created on the fly by this notebook)
# ============================================================================
contract_business_unit = "Testing"
contract_use_case_name = "ImageModels"
contract_environment   = "DEV"

# ============================================================================
# 🔁 azd ENVIRONMENT OVERRIDES
# ============================================================================
if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group": ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":       ["AZURE_LOCATION", "LOCATION"],
        "llm_backends":   (["LLM_BACKEND_CONFIG", "LLM_BACKENDS_CONFIG"], "json"),
    }, verbose=False)
    if "resource_group" in loaded:  governance_hub_resource_group = loaded["resource_group"]
    if "location" in loaded:        location = loaded["location"]
    if "llm_backends" in loaded:    llm_backends_config = loaded["llm_backends"]
    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")
    utils.print_ok(f"Backends loaded: {len(llm_backends_config)}")

utils.print_ok("Notebook variables initialized!")

### 1️⃣ Verify Azure CLI and Connected Subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

### 2️⃣ Initialize APIM Client Tool

Discover the Unified AI API and retrieve gateway configuration:

In [ ]:
try:
    apimClientTool = APIMClientTool(governance_hub_resource_group)
    apimClientTool.initialize()
    apimClientTool.discover_api("unified-ai")

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint = str(apimClientTool.azure_endpoint)

    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models: {supported_models}")
    utils.print_ok("Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

### 2️⃣B LLM Backend Contract Onboarding (FLUX & MAI image backends)

`gpt-image-1.5` works out of the box on your existing `ai-foundry` backend — it needs no special backend type. **FLUX** and **MAI** require dedicated `azure-flux` / `azure-mai` backends that point at the Foundry resource's native image surfaces (`/providers/blackforestlabs/v1/...` and `/mai/v1/images/...`).

This step augments the `llm_backends_config` loaded from your `azd` environment with:
- `gpt-image-1.5` added to the existing `ai-foundry` backend (only if missing)
- an `azure-flux` backend carrying the `modelPath` slug (e.g. `flux-2-pro`)
- an `azure-mai` backend (model travels in the request body)

then redeploys the LLM Backend Onboarding Bicep so the routing + `metadata-config` pick up the new image models. Skipped when `run_backend_onboarding = False`, or when neither FLUX nor MAI is enabled and `gpt-image` is already onboarded.

In [ ]:
import copy

# Discover the user-assigned managed identity used by the onboarding contract
mi = apimClientTool.get_managed_identity_info()
managed_identity_name           = mi.get('name')
managed_identity_resource_group = mi.get('resourceGroup') or governance_hub_resource_group
utils.print_info(f"Managed Identity: {managed_identity_name} (rg={managed_identity_resource_group})")


def _derive_image_endpoint(cfg):
    """Build the Foundry image endpoint (.services.ai.azure.com) from an existing ai-foundry backend."""
    for b in (cfg or []):
        if b.get('backendType') == 'ai-foundry' and b.get('endpoint'):
            host = b['endpoint'].split('//', 1)[-1].split('/', 1)[0]   # e.g. aif-xxx.cognitiveservices.azure.com
            account = host.split('.', 1)[0]                            # aif-xxx
            return f"https://{account}.services.ai.azure.com/"
    return ""


def _norm_models(models):
    return [dict(m) if isinstance(m, dict) else {"name": m} for m in models]


img_endpoint = image_foundry_endpoint or _derive_image_endpoint(llm_backends_config)
onboard_cfg = copy.deepcopy(llm_backends_config) if llm_backends_config else []

# 1) gpt-image rides the existing ai-foundry backend — ensure the model is present
if test_openai_image:
    target = next((b for b in onboard_cfg if b.get('backendType') == 'ai-foundry'), None)
    if target is None:
        utils.print_warning("No ai-foundry backend found — gpt-image must already be onboarded separately.")
    else:
        target['supportedModels'] = _norm_models(target.get('supportedModels', []))
        names = [m['name'] for m in target['supportedModels']]
        if openai_image_model not in names:
            target['supportedModels'].append({"name": openai_image_model, "modelFormat": "OpenAI", "modelVersion": "1"})
            utils.print_ok(f"Added '{openai_image_model}' to ai-foundry backend '{target['backendId']}'.")
        else:
            utils.print_info(f"'{openai_image_model}' already present on ai-foundry backend '{target['backendId']}'.")

# 2) FLUX — dedicated azure-flux backend (requires a modelPath slug)
if test_flux:
    if not img_endpoint:
        utils.print_warning("No image Foundry endpoint resolved — set image_foundry_endpoint for FLUX.")
    onboard_cfg.append({
        "backendId": "aif-citadel-flux",
        "backendType": "azure-flux",
        "endpoint": img_endpoint,
        "authType": "managed-identity",
        "supportedModels": [
            {"name": flux_image_model, "modelPath": flux_image_model_path, "modelFormat": "BlackForestLabs", "modelVersion": "1"}
        ],
        "priority": 1,
        "weight": 100,
    })
    utils.print_ok(f"Queued azure-flux backend for '{flux_image_model}' (modelPath '{flux_image_model_path}').")

# 3) MAI — dedicated azure-mai backend (model in body)
if test_mai:
    if not img_endpoint:
        utils.print_warning("No image Foundry endpoint resolved — set image_foundry_endpoint for MAI.")
    onboard_cfg.append({
        "backendId": "aif-citadel-mai",
        "backendType": "azure-mai",
        "endpoint": img_endpoint,
        "authType": "managed-identity",
        "supportedModels": [
            {"name": mai_image_model, "modelFormat": "MAI", "modelVersion": "1"}
        ],
        "priority": 1,
        "weight": 100,
    })
    utils.print_ok(f"Queued azure-mai backend for '{mai_image_model}'.")


def _fmt_model(m):
    parts = [f"name: '{m['name']}'"]
    for k in ('sku', 'modelFormat', 'modelVersion', 'retirementDate', 'apiVersion', 'inferenceApiVersion', 'modelPath'):
        if m.get(k) not in (None, ''):
            parts.append(f"{k}: '{m[k]}'")
    for k in ('capacity', 'timeout'):
        if m.get(k) not in (None, ''):
            parts.append(f"{k}: {m[k]}")
    return "{ " + ", ".join(parts) + " }"


def _fmt_backend(b):
    models = "\n      ".join(_fmt_model(m if isinstance(m, dict) else {"name": m}) for m in b['supportedModels'])
    auth = f"    authType: '{b['authType']}'\n" if b.get('authType') else ""
    return (f"  {{\n    backendId: '{b['backendId']}'\n    backendType: '{b['backendType']}'\n"
            f"    endpoint: '{b['endpoint']}'\n{auth}    supportedModels: [\n      {models}\n    ]\n"
            f"    priority: {b.get('priority', 1)}\n    weight: {b.get('weight', 100)}\n  }}")


if not run_backend_onboarding:
    utils.print_warning("run_backend_onboarding = False — skipping FLUX/MAI backend onboarding.")
elif not (test_flux or test_mai):
    utils.print_info("Neither FLUX nor MAI enabled — no special backend onboarding required (gpt-image rides ai-foundry).")
elif not onboard_cfg:
    utils.print_warning("llm_backends_config is empty (no azd LLM_BACKEND_CONFIG) — paste it manually before onboarding.")
else:
    onboarding_dir = "../bicep/infra/llm-backend-onboarding"
    onboarding_params = os.path.join(onboarding_dir, "llm-backends-image-validation.bicepparam")
    backends_str = "\n".join(_fmt_backend(b) for b in onboard_cfg)

    onboarding_content = f"""using './main.bicep'

// Generated by citadel-image-models-tests.ipynb @ {time.strftime('%Y-%m-%d %H:%M:%S')}
// Existing backends (from azd) + dedicated azure-flux / azure-mai image backends.

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

param llmBackendConfig = [
{backends_str}
]

param configureCircuitBreaker = true
"""

    with open(onboarding_params, 'w') as fh:
        fh.write(onboarding_content)
    utils.print_ok(f"Wrote {onboarding_params}")
    print("\n" + "-" * 60)
    print(onboarding_content)
    print("-" * 60)

    onboarding_name = f"image-backends-{time.strftime('%Y%m%d%H%M%S')}"
    onboarding_template = os.path.join(onboarding_dir, "main.bicep")
    onboarding_cmd = (
        f"az deployment sub create --name {onboarding_name} "
        f"--location {location} --template-file {onboarding_template} --parameters {onboarding_params}"
    )
    out = utils.run(onboarding_cmd, f"Onboarding '{onboarding_name}' succeeded", f"Onboarding '{onboarding_name}' failed")
    if out.success:
        utils.print_ok("✅ Image backends onboarded — FLUX/MAI routing + metadata-config refreshed.")
        # Refresh discovered models so later cells see the new image models
        apimClientTool.discover_api("unified-ai")

### 3️⃣ Provision Access Contract

Deploy an access contract via Bicep to create an APIM product + subscription scoped to the **enabled image models**. The product policy uses `set-llm-requested-model` (which now also resolves the `x-ai-model` header for multipart edits) and enforces `allowedModels`.

In [ ]:
bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

timestamp = time.strftime('%Y%m%d%H%M%S')
contract_name = f"image-models-test-{timestamp}"

folder_name = f"{contract_business_unit.lower()}-{contract_use_case_name.lower()}"
environment_folder = contract_environment.lower()
contract_folder = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(contract_folder, exist_ok=True)
utils.print_info(f"📁 Created folder: {contract_folder}")

# Build allowedModels from the enabled image models
allowed_models_list = []
if test_openai_image: allowed_models_list.append(openai_image_model)
if test_flux:         allowed_models_list.append(flux_image_model)
if test_mai:          allowed_models_list.append(mai_image_model)
if not allowed_models_list:
    raise Exception("Enable at least one of test_openai_image / test_flux / test_mai.")
allowed_models_csv = ",".join(allowed_models_list)
utils.print_info(f"Allowed models for access contract: {allowed_models_csv}")

product_policy = f'''<policies>
    <inbound>
        <base />
        <!-- Extract and validate model parameter (body / path / x-ai-model header) -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Restrict access to only the image models defined in notebook parameters -->
        <set-variable name="allowedModels" value="{allowed_models_csv}" />

        <!-- Enable advanced response headers offered by set-response-headers fragment -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''

policy_file_dest = os.path.join(contract_folder, "ai-product-policy.xml")
with open(policy_file_dest, 'w') as f:
    f.write(product_policy)
utils.print_ok(f"📋 Custom policy file created: {policy_file_dest}")

params_file = os.path.join(contract_folder, "main.bicepparam")
params_content = f"""using '../../../main.bicep'

// Image Models Test Contract - Generated from Notebook
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  name: 'placeholder'
}}

param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{contract_business_unit}'
  useCaseName: '{contract_use_case_name}'
  environment: '{contract_environment}'
}}

param apiNameMapping = {{
  LLM: ['unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: 'IMAGE-MODELS-TEST-ENDPOINT'
    apiKeySecretName: 'IMAGE-MODELS-TEST-KEY'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Image models test contract - generated from validation notebook'

param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
"""

with open(params_file, 'w') as f:
    f.write(params_content)
utils.print_ok(f"✅ Parameter file created: {params_file}")

product_id = f"LLM-{contract_business_unit}-{contract_use_case_name}-{contract_environment}"
utils.print_info(f"Deploying access contract: {contract_name} (Product: {product_id})...")

deployment_cmd = f"az deployment sub create --name {contract_name} --location {location} --template-file {template_file} --parameters {params_file}"
output = utils.run(deployment_cmd, f"Deployment '{contract_name}' succeeded", f"Deployment '{contract_name}' failed")

if output.success:
    utils.print_ok(f"✅ Access contract deployed! Product ID: {product_id}")
else:
    utils.print_error("❌ Access contract deployment failed!")

apimClientTool.initialize()

### 4️⃣ Retrieve API Key & Build Endpoints

In [ ]:
subscription_name = f"{product_id}-SUB-01"

api_key = None
for sub in apimClientTool.apim_subscriptions:
    if subscription_name.lower() in sub.get('name', '').lower():
        api_key = sub.get('key')
        utils.print_ok(f"Found API key from subscription: {sub.get('name')}")
        break

if not api_key:
    raise Exception(f"No API key found for subscription '{subscription_name}'. Ensure the access contract was deployed successfully.")

base_url = azure_endpoint.rstrip('/')
images_generations_url = f"{base_url}/unified-ai/v1/images/generations"
images_edits_url       = f"{base_url}/unified-ai/v1/images/edits"
deployments_url        = f"{base_url}/unified-ai/deployments"

def deployments_generations_url(model):
    return f"{base_url}/unified-ai/openai/deployments/{model}/images/generations"

utils.print_info(f"Images (generations): {images_generations_url}")
utils.print_info(f"Images (edits):       {images_edits_url}")
utils.print_info(f"Deployments:          {deployments_url}")
utils.print_ok("API key and endpoints configured successfully!")

### 5️⃣ Discover Deployments

Confirm the enabled image models are visible (and granted by the access contract):

In [ ]:
try:
    response = requests.get(deployments_url, headers={'api-key': api_key}, timeout=30)
    utils.print_response_code(response)
    if response.status_code == 200:
        data = json.loads(response.text)
        names = [d.get('name', 'unknown') for d in data.get('value', [])]
        utils.print_ok(f"Allowed deployments: {names}")
        for m in allowed_models_list:
            if m in names:
                utils.print_ok(f"  • {m} is discoverable ✅")
            else:
                utils.print_warning(f"  • {m} NOT found in discovery — check onboarding/allowedModels")
    else:
        utils.print_error(f"Error listing deployments: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

### 6️⃣ Image Generation Helper

Posts an OpenAI-style image generation request and asserts an OpenAI-shaped response (`data[0].b64_json` or `data[0].url`).

In [ ]:
# Tracks the on-disk path of each saved generated image, keyed by label — used by the edit test below.
generated_image_files = {}

def generate_image(model, prompt, extra=None, url=None, label=None):
    label = label or model
    target = url or images_generations_url
    payload = {"model": model, "prompt": prompt, "n": 1}
    if extra:
        payload.update(extra)
    utils.print_info(f"[{label}] POST {target}")
    try:
        response = requests.post(target, headers={'api-key': api_key, 'Content-Type': 'application/json'}, json=payload, timeout=120)
        utils.print_response_code(response)
        if response.status_code != 200:
            utils.print_error(f"[{label}] Failed: {response.text[:400]}")
            return False
        data = json.loads(response.text)
        items = data.get('data', [])
        if not items:
            utils.print_error(f"[{label}] Response has no 'data' array: {response.text[:300]}")
            return False
        b64 = items[0].get('b64_json')
        if b64:
            utils.print_ok(f"[{label}] Received b64_json image ({len(b64)} chars)")
            if save_images:
                fname = f"image-test-{label.replace('.', '').replace('/', '')}.png"
                with open(fname, 'wb') as fh:
                    fh.write(base64.b64decode(b64))
                generated_image_files[label] = os.path.abspath(fname)
                utils.print_info(f"[{label}] Saved {fname}")
            return True
        if items[0].get('url'):
            utils.print_ok(f"[{label}] Received image URL: {items[0]['url'][:80]}...")
            return True
        utils.print_error(f"[{label}] No b64_json or url in response: {response.text[:300]}")
        return False
    except Exception as e:
        utils.print_error(f"[{label}] Request error: {e}")
        return False

### 7️⃣ gpt-image Generation (body model + deployments path)

Tests both the unified `/v1/images/generations` (model in body) and the Azure OpenAI `/openai/deployments/{model}/images/generations` (model in URL) forms.

In [ ]:
if test_openai_image:
    # Body-model form (OpenAI SDK style)
    generate_image(openai_image_model, image_prompt, extra={"size": "1024x1024", "quality": "medium"}, label=f"{openai_image_model} (body)")
    # Deployments-path form (Azure OpenAI SDK style)
    generate_image(openai_image_model, image_prompt, extra={"size": "1024x1024"}, url=deployments_generations_url(openai_image_model), label=f"{openai_image_model} (deployments)")
else:
    utils.print_warning("test_openai_image is False — skipping gpt-image generation.")

### 8️⃣ FLUX Generation (azure-flux, optional)

Routes to the native Black Forest Labs surface (`/providers/blackforestlabs/v1/{modelPath}?api-version=preview`). Requires a `modelPath` slug onboarded for the model.

In [ ]:
if test_flux:
    generate_image(flux_image_model, image_prompt, extra={"width": 1024, "height": 1024}, label=flux_image_model)
else:
    utils.print_warning("test_flux is False — skipping FLUX generation.")

### 9️⃣ MAI Generation (azure-mai, optional)

Routes to the native Microsoft MAI surface (`/mai/v1/images/generations`).

In [ ]:
if test_mai:
    generate_image(mai_image_model, image_prompt, extra={"width": 1024, "height": 1024}, label=mai_image_model)
else:
    utils.print_warning("test_mai is False — skipping MAI generation.")

### 🔟 Missing-Model Error (multipart edits without `x-ai-model`)

`POST /unified-ai/v1/images/edits` as multipart without the `x-ai-model` header must return `400 missing_model_parameter` with guidance to set the header.

In [ ]:
# Send a small multipart body WITHOUT the x-ai-model header
files = {
    'prompt': (None, image_prompt),
    'image': ('blank.png', base64.b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=='), 'image/png'),
}
try:
    response = requests.post(images_edits_url, headers={'api-key': api_key}, files=files, timeout=60)
    utils.print_response_code(response)
    if response.status_code == 400 and 'missing_model_parameter' in response.text:
        utils.print_ok("Correctly rejected with 400 missing_model_parameter ✅")
        utils.print_info(response.text[:300])
    else:
        utils.print_warning(f"Unexpected response: {response.status_code} {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

### 🔟B Image Edit on a Generated Image (multipart)

Takes an image produced by the generation tests and edits it via `POST /unified-ai/openai/deployments/{model}/images/edits` — the recommended multipart-edit form (model in the URL, so no `x-ai-model` header is needed). The edited result is saved next to the notebook for visual inspection.

> `gpt-image` supports edits. Run the gpt-image generation cell first (with `save_images = True`) so a source image exists.

In [ ]:
def edit_image(model, image_path, prompt, use_header=False):
    if not image_path or not os.path.exists(image_path):
        utils.print_warning(f"No source image at '{image_path}' — run the generation tests first (save_images=True).")
        return False
    if use_header:
        # Unified surface: model can't be read from a multipart form field, so route via x-ai-model header.
        target = images_edits_url
        headers = {'api-key': api_key, 'x-ai-model': model}
        form = {'prompt': (None, prompt), 'model': (None, model)}
    else:
        # Recommended form: model in the URL (deployments path).
        target = f"{base_url}/unified-ai/openai/deployments/{model}/images/edits"
        headers = {'api-key': api_key}
        form = {'prompt': (None, prompt)}
    utils.print_info(f"[edit:{model}] POST {target}")
    try:
        with open(image_path, 'rb') as fh:
            files = dict(form)
            files['image'] = (os.path.basename(image_path), fh.read(), 'image/png')
        response = requests.post(target, headers=headers, files=files, timeout=180)
        utils.print_response_code(response)
        if response.status_code != 200:
            utils.print_error(f"[edit:{model}] Failed: {response.text[:400]}")
            return False
        data = json.loads(response.text)
        items = data.get('data', [])
        b64 = items[0].get('b64_json') if items else None
        if b64:
            out = f"image-edit-{model.replace('.', '').replace('/', '')}.png"
            with open(out, 'wb') as oh:
                oh.write(base64.b64decode(b64))
            utils.print_ok(f"[edit:{model}] Edited image saved → {out}")
            return True
        utils.print_error(f"[edit:{model}] No b64_json in response: {response.text[:300]}")
        return False
    except Exception as e:
        utils.print_error(f"[edit:{model}] Request error: {e}")
        return False


edit_prompt = "Add a light snowfall and a soft blue winter tone to the scene"
if test_openai_image:
    # Use an image saved by the gpt-image generation test (body form first, else deployments form)
    src = generated_image_files.get(f"{openai_image_model} (body)") \
        or generated_image_files.get(f"{openai_image_model} (deployments)")
    edit_image(openai_image_model, src, edit_prompt)
else:
    utils.print_warning("test_openai_image is False — skipping image edit (gpt-image supports edits).")

### 1️⃣1️⃣ RBAC Enforcement (disallowed model → 403)

Requesting an image model that is NOT in the access contract's `allowedModels` must be rejected.

In [ ]:
payload = {"model": disallowed_image_model, "prompt": image_prompt, "n": 1, "size": "1024x1024"}
try:
    response = requests.post(images_generations_url, headers={'api-key': api_key, 'Content-Type': 'application/json'}, json=payload, timeout=60)
    utils.print_response_code(response)
    if response.status_code == 403:
        utils.print_ok(f"Disallowed model '{disallowed_image_model}' correctly blocked with 403 ✅")
    else:
        utils.print_warning(f"Expected 403 for disallowed model, got {response.status_code}: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## ✅ Summary

This notebook validated image generation/editing through the Unified AI API:
- Onboarded dedicated `azure-flux` / `azure-mai` image backends (gpt-image rides the existing `ai-foundry` backend)
- gpt-image generation via body-model and deployments-path forms
- FLUX (`azure-flux`) and MAI (`azure-mai`) native surfaces (when enabled)
- A real image edit on a generated image via the deployments-path multipart form
- Multipart-edit missing-model guidance (`x-ai-model` header requirement)
- RBAC enforcement via the access contract's `allowedModels`

See [LLM Access Guide — Image Models](../guides/llm-access-guide.md#image-models) and [LLM Backend Onboarding — Image Models](../bicep/infra/llm-backend-onboarding/README.md#image-models) for details.